# EDA - features and target for the reject-effectiveness model

Before training anything: check that the target has real variance (not
99% one class), and that each candidate feature actually differs between
the two outcomes -- if a feature looks identical whether reject worked or
not, it won't help the model.

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

df = pd.read_csv('../clean_data/dataset_completed_clean.csv')
print('Full shape:', df.shape)

Full shape: (4100, 31)


## Step 1 - Filter to the subset where the differential test ran

In [2]:
df_model = df[df['reject_click_attempted'] == True].copy()
print(f'Rows with a usable differential test: {len(df_model)} / {len(df)}')

Rows with a usable differential test: 1583 / 4100


## Step 2 - Build and check the target
This is the single most important check before anything else: is the
split reasonably balanced, or is one class so rare the model could just
predict the majority every time and look 'accurate'?

In [3]:
df_model['reject_works'] = df_model['tracker_count_after_reject'] == 0

print(df_model['reject_works'].value_counts())
print()
print(df_model['reject_works'].value_counts(normalize=True).round(3) * 100)

reject_works
False    899
True     684
Name: count, dtype: int64

reject_works
False    56.8
True     43.2
Name: proportion, dtype: float64


## Step 3 - Build the derived hosting feature

In [4]:
MAJOR_CLOUD_CDN = ['amazon', 'google', 'fastly', 'akamai', 'cloudflare', 'microsoft']

def is_major_cloud_cdn(org):
    if pd.isna(org):
        return False
    org_lower = org.lower()
    return any(name in org_lower for name in MAJOR_CLOUD_CDN)

df_model['uses_major_cloud_cdn'] = df_model['org'].apply(is_major_cloud_cdn)
print(df_model['uses_major_cloud_cdn'].value_counts(dropna=False))

uses_major_cloud_cdn
False    993
True     590
Name: count, dtype: int64


## Step 4 - Null check on ALL candidate features, within this subset
Nulls behave differently once filtered to this smaller subset -- re-check
rather than assume the full-dataset percentages still apply.

In [5]:
feature_cols = [
    'cmp_detected', 'tracker_count', 'ad_network_count',
    'cookies_before_interaction', 'non_essential_cookies_before_interaction',
    'long_lived_cookies_count', 'has_privacy_policy',
    'is_hosting', 'uses_major_cloud_cdn'
]

null_pct = (df_model[feature_cols].isnull().sum() / len(df_model) * 100).sort_values(ascending=False)
print(null_pct.round(1))

has_privacy_policy                          0.2
is_hosting                                  0.2
cmp_detected                                0.0
tracker_count                               0.0
ad_network_count                            0.0
cookies_before_interaction                  0.0
non_essential_cookies_before_interaction    0.0
long_lived_cookies_count                    0.0
uses_major_cloud_cdn                        0.0
dtype: float64


## Step 5 - Numeric features: do they actually differ by outcome?
Groups by reject_works (True/False), shows mean/median for each numeric
feature. A useful feature should show a visible difference between groups.

In [6]:
numeric_features = ['tracker_count', 'ad_network_count', 'cookies_before_interaction',
                     'non_essential_cookies_before_interaction', 'long_lived_cookies_count']

print(df_model.groupby('reject_works')[numeric_features].mean().round(2))
print()
print(df_model.groupby('reject_works')[numeric_features].median().round(2))

              tracker_count  ad_network_count  cookies_before_interaction  non_essential_cookies_before_interaction  long_lived_cookies_count
reject_works                                                                                                                                 
False                 18.89              9.37                        8.37                                      7.58                      1.54
True                  11.44              4.25                        5.61                                      5.00                      1.12

              tracker_count  ad_network_count  cookies_before_interaction  non_essential_cookies_before_interaction  long_lived_cookies_count
reject_works                                                                                                                                 
False                  16.0               7.0                         6.0                                       5.0                       1.0
True 

## Step 6 - Categorical/boolean features: do they differ by outcome?

In [7]:
bool_features = ['has_privacy_policy', 'is_hosting', 'uses_major_cloud_cdn']

for col in bool_features:
    print(f'--- {col} ---')
    print(pd.crosstab(df_model[col], df_model['reject_works'], normalize='index').round(3))
    print()

--- has_privacy_policy ---
reject_works        False  True 
has_privacy_policy              
False               0.580  0.420
True                0.566  0.434

--- is_hosting ---
reject_works  False  True 
is_hosting                
False         0.493  0.507
True          0.598  0.402

--- uses_major_cloud_cdn ---
reject_works          False  True 
uses_major_cloud_cdn              
False                 0.533  0.467
True                  0.627  0.373



## Step 7 - cmp_detected: reject-works rate by vendor
This is likely your strongest feature -- check it's not just noise from
vendors with only 1-2 sites each.

In [8]:
cmp_summary = df_model.groupby('cmp_detected').agg(
    n_sites=('reject_works', 'count'),
    pct_reject_works=('reject_works', 'mean')
).sort_values('n_sites', ascending=False)

cmp_summary['pct_reject_works'] = (cmp_summary['pct_reject_works'] * 100).round(1)
print(cmp_summary)

                n_sites  pct_reject_works
cmp_detected                             
none                576              61.3
onetrust            387              11.1
didomi              132              39.4
usercentrics         95              80.0
trustarc             66              47.0
cookiebot            64               7.8
sourcepoint          54              59.3
iubenda              51              29.4
cookie notice        36              13.9
cookieyes            28              85.7
trustcommander       23              56.5
complianz            15              60.0
quantcast            14              35.7
klaro                10              80.0
osano                10              20.0
fundingchoices       10              10.0
axeptio               8              87.5
commanders act        3              66.7
setupad               1             100.0


## Step 8 - Distribution shape of tracker_count (before)
Check for skew/outliers that might need a log transform or capping later.

In [9]:
print(df_model['tracker_count'].describe())
print()
print('Top 5 highest tracker_count rows:')
print(df_model.nlargest(5, 'tracker_count')[['url', 'tracker_count', 'reject_works']])

count    1583.000000
mean       15.674668
std        11.660968
min         1.000000
25%         8.000000
50%        12.000000
75%        20.000000
max       114.000000
Name: tracker_count, dtype: float64

Top 5 highest tracker_count rows:
                         url  tracker_count  reject_works
3378     https://wowhead.com          114.0         False
3957      https://sfgate.com           96.0         False
3114        https://ndtv.com           93.0         False
887   https://fcinter1908.it           86.0          True
3296    https://cricbuzz.com           82.0         False


## Check 1 - Why does 'none' (no recognized CMP) still work 61.3% of the time?
Compare the 'none' group's tracker_count against the rest -- if it's
mostly low-tracker sites, that explains the decent success rate without
needing a CMP vendor at all (custom banners on simple sites).

In [10]:
df_none = df_model[df_model['cmp_detected'] == 'none']
df_known_cmp = df_model[df_model['cmp_detected'] != 'none']

print('=== "none" group (n=%d) ===' % len(df_none))
print(df_none[['tracker_count', 'ad_network_count']].describe().round(2))
print()
print('=== known CMP group (n=%d) ===' % len(df_known_cmp))
print(df_known_cmp[['tracker_count', 'ad_network_count']].describe().round(2))

=== "none" group (n=576) ===
       tracker_count  ad_network_count
count         576.00            576.00
mean           11.54              5.22
std            10.10              6.91
min             1.00              0.00
25%             5.00              1.00
50%             9.00              3.00
75%            14.00              6.00
max            96.00             71.00

=== known CMP group (n=1007) ===
       tracker_count  ad_network_count
count        1007.00           1007.00
mean           18.04              8.27
std            11.84              8.07
min             1.00              0.00
25%            10.00              3.00
50%            15.00              6.00
75%            23.00             11.00
max           114.00             67.00


In [11]:
print('Median tracker_count, none vs known CMP:')
print('  none:      ', df_none['tracker_count'].median())
print('  known CMP: ', df_known_cmp['tracker_count'].median())
print()
print('reject_works rate, none vs known CMP:')
print('  none:      ', (df_none['reject_works'].mean() * 100).round(1), '%')
print('  known CMP: ', (df_known_cmp['reject_works'].mean() * 100).round(1), '%')

Median tracker_count, none vs known CMP:
  none:       9.0
  known CMP:  15.0

reject_works rate, none vs known CMP:
  none:       61.3 %
  known CMP:  32.9 %


## Check 2 - Correlation between numeric features
If tracker_count and ad_network_count move almost perfectly together,
one may be redundant (though RandomForest tolerates this fine either way).

In [12]:
numeric_features = ['tracker_count', 'ad_network_count', 'cookies_before_interaction',
                     'non_essential_cookies_before_interaction', 'long_lived_cookies_count']

corr_matrix = df_model[numeric_features].corr().round(2)
print(corr_matrix)

                                          tracker_count  ad_network_count  cookies_before_interaction  non_essential_cookies_before_interaction  long_lived_cookies_count
tracker_count                                      1.00              0.93                        0.50                                      0.50                      0.47
ad_network_count                                   0.93              1.00                        0.45                                      0.45                      0.42
cookies_before_interaction                         0.50              0.45                        1.00                                      0.99                      0.65
non_essential_cookies_before_interaction           0.50              0.45                        0.99                                      1.00                      0.66
long_lived_cookies_count                           0.47              0.42                        0.65                                      0.66       

## Check 3 - Within OneTrust specifically: is low success rate about the
vendor itself, or just that OneTrust sites tend to have more trackers?
Compare tracker_count for OneTrust's True vs False rows -- if it still
shows a gap, tracker_count matters even within the same vendor (both
effects are real, independently).

In [13]:
df_onetrust = df_model[df_model['cmp_detected'] == 'onetrust']

print('OneTrust sites where reject WORKED (n=%d):' % (df_onetrust['reject_works'] == True).sum())
print(df_onetrust[df_onetrust['reject_works'] == True]['tracker_count'].describe().round(2))
print()
print('OneTrust sites where reject did NOT work (n=%d):' % (df_onetrust['reject_works'] == False).sum())
print(df_onetrust[df_onetrust['reject_works'] == False]['tracker_count'].describe().round(2))

OneTrust sites where reject WORKED (n=43):
count    43.00
mean     11.79
std       6.35
min       4.00
25%       7.50
50%      11.00
75%      14.00
max      30.00
Name: tracker_count, dtype: float64

OneTrust sites where reject did NOT work (n=344):
count    344.00
mean      18.38
std       12.07
min        3.00
25%       11.00
50%       14.50
75%       23.00
max      114.00
Name: tracker_count, dtype: float64


## Check 3b - Same comparison for a HIGH-performing vendor (CookieYes)
as a point of contrast.

In [14]:
df_cookieyes = df_model[df_model['cmp_detected'] == 'cookieyes']

print('CookieYes sites where reject WORKED (n=%d):' % (df_cookieyes['reject_works'] == True).sum())
print(df_cookieyes[df_cookieyes['reject_works'] == True]['tracker_count'].describe().round(2))
print()
print('CookieYes sites where reject did NOT work (n=%d):' % (df_cookieyes['reject_works'] == False).sum())
print(df_cookieyes[df_cookieyes['reject_works'] == False]['tracker_count'].describe().round(2))

CookieYes sites where reject WORKED (n=24):
count    24.00
mean     12.92
std       5.35
min       3.00
25%       8.75
50%      13.00
75%      16.25
max      23.00
Name: tracker_count, dtype: float64

CookieYes sites where reject did NOT work (n=4):
count     4.00
mean     25.75
std      13.67
min      16.00
25%      19.00
50%      20.50
75%      27.25
max      46.00
Name: tracker_count, dtype: float64


# Save the ML-ready dataset (reject-effectiveness model)

## trimmed feature set + target + id columns
Dropped ad_network_count (0.93 correlated with tracker_count) and
cookies_before_interaction (0.99 correlated with non_essential_cookies_before_interaction).

In [15]:
final_cols = [
    'url', 'domain',
    'cmp_detected',
    'tracker_count',
    'non_essential_cookies_before_interaction',
    'long_lived_cookies_count',
    'has_privacy_policy',
    'is_hosting',
    'uses_major_cloud_cdn',
    'reject_works'
]

final_cols = [c for c in final_cols if c in df_model.columns]
df_final = df_model[final_cols].copy()

print('Final shape:', df_final.shape)
print('Columns:', df_final.columns.tolist())
df_final.head()

Final shape: (1583, 10)
Columns: ['url', 'domain', 'cmp_detected', 'tracker_count', 'non_essential_cookies_before_interaction', 'long_lived_cookies_count', 'has_privacy_policy', 'is_hosting', 'uses_major_cloud_cdn', 'reject_works']


,url,domain,cmp_detected,tracker_count,non_essential_cookies_before_interaction,long_lived_cookies_count,has_privacy_policy,is_hosting,uses_major_cloud_cdn,reject_works
0,https://telekom.de,telekom.de,none,3.0,5.0,0.0,True,True,False,False
2,https://bbc.co.uk,bbc.co.uk,none,25.0,16.0,6.0,True,True,True,False
3,https://www.gov.uk,gov.uk,none,4.0,1.0,0.0,True,NaN,False,True
4,https://amazon.co.uk,amazon.co.uk,cookie notice,8.0,10.0,0.0,True,True,True,True
5,https://dailymail.co.uk,dailymail.co.uk,trustarc,30.0,15.0,4.0,True,False,True,False


In [16]:
df_final.to_csv('../clean_data/final_model_data_ready_to_train.csv', index=False)
print(f'Saved csv — {df_final.shape}')

Saved csv — (1583, 10)
